In [ ]:
import Pkg

c = ["CSV", "Tidier", "DataFrames", "Metal", "Distributions", "RCall"]
Pkg.add(c)

using CSV, DataFrames, Tidier, Metal, Distributions, RCall

# 座標データが時計回りかどうかを判定
function is_clockwise(df::DataFrame)::Bool

    x = df.X
    y = df.Y
    x_next = circshift(x, -1)
    y_next = circshift(y, -1)
    
    signed_area = sum((x_next .- x) .* (y_next .+ y))

    return signed_area > 0.0

end

# PolygonIDごとに座標データを取り出し，反時計回りになるようにする
function to_counterclockwise(df)::DataFrame

    if is_clockwise(df)
        df = reverse(df)
    end

    return df

end

# CSVファイルから座標データを読み込む
function reading_coordinate(filepath::String)::Tuple{DataFrame, DataFrame}
    # CSVファイルを読み込む
    polygon_df = CSV.read(filepath, DataFrame)

    # PolygonIDごとに座標データを反時計回りになるよう変換
    clockwise_corrected_df = correct_polygon_clockwise(polygon_df)

    # 外周ポリゴンデータと障害物ポリゴンデータを抽出
    outer_df, obstacle_df = extract_polygon_data(clockwise_corrected_df)

    return (outer_df, obstacle_df)

end

# PolygonIDごとに座標データを反時計回りに変換する
function correct_polygon_clockwise(polygon_df::DataFrame)::DataFrame

    correct_df = DataFrame(
        X = Float32[],
        Y = Float32[],
        PolygonID = Int32[]
    )

    for polygon_id in unique(polygon_df.PolygonID)
        correct_df = @chain polygon_df begin
            # @filter(:PolygonID == polygon_id)
            @filter(PolygonID == !!polygon_id)
            to_counterclockwise(_)
            vcat(correct_df, _)
        end
    end

    return correct_df
    
end

# 外周ポリゴンデータと障害物ポリゴンデータを抽出
function extract_polygon_data(polygon_df::DataFrame)::Tuple{DataFrame, DataFrame}

    min_polygon_id = minimum(polygon_df.PolygonID)

    outer_df = @chain polygon_df begin
        @filter(PolygonID == !!min_polygon_id)
        @rename(x0 = X, y0 = Y, id = PolygonID)
        @mutate(id = Int32(id + 1))
        # @filter(:PolygonID == min_polygon_id)
        # @rename(:X => :x0, :Y => :y0, :PolygonID => :id)
        # @mutate(:id = Int32(:id + 1))
    end

    obstacle_df = @chain polygon_df begin
        @filter(PolygonID != !!min_polygon_id)
        @rename(x0 = X, y0 = Y, id = PolygonID)
        @mutate(id = Int32(id + 1))
    end

    return (outer_df, obstacle_df)
    
end

# 外周および遮蔽物を構成する
# 辺を取得する
# 始点 (x0, y0）
# 終点 (x1, y1)
function polygon_to_edges(polygon_df::DataFrame)::DataFrame

    edge_df = @chain polygon_df begin
        groupby(:id)
        transform(
            :x0 => (x -> circshift(x, -1)) => :x1,
            :y0 => (y -> circshift(y, -1)) => :y1
        )
    end

    return(edge_df)

end

# カーネル関数を使った3次元配列代入 #1
function kernel_assign_1(vec0_x::MtlDeviceArray{Float32, 3}, vec0_y::MtlDeviceArray{Float32, 3},
                            vec1_x::MtlDeviceArray{Float32, 3}, vec1_y::MtlDeviceArray{Float32, 3},
                            xx::MtlDeviceArray{Float32, 1}, yy::MtlDeviceArray{Float32, 1},
                            x0::MtlDeviceArray{Float32, 1}, y0::MtlDeviceArray{Float32, 1},
                            x1::MtlDeviceArray{Float32, 1}, y1::MtlDeviceArray{Float32, 1})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z
                        
    if i > size(vec0_x, 1) || j > size(vec0_x, 2) || k > size(vec0_x, 3)
        return nothing
    end
                        
    vec0_x[i, j, k] = x0[k] - xx[i]
    vec1_x[i, j, k] = x1[k] - xx[i]
    vec0_y[i, j, k] = y0[k] - yy[j]
    vec1_y[i, j, k] = y1[k] - yy[j]
                        
    return nothing

end

# 内外判定
function in_out_judge(xx::Vector{Float32}, yy::Vector{Float32}, poly_edge::DataFrame)::DataFrame

    n_x, n_y = length(xx), length(yy)
    poly_ids = unique(poly_edge.id)
    in_out = Array{Bool}(undef, n_x, n_y, length(poly_ids))

    for (p_idx, p) in enumerate(poly_ids)
        edge =
        @chain poly_edge begin
            @filter(id == !!p)
            @select(x0, y0, x1, y1)
        end

        Mtl_x0 = MtlArray{Float32}(edge.x0)
        Mtl_y0 = MtlArray{Float32}(edge.y0)
        Mtl_x1 = MtlArray{Float32}(edge.x1)
        Mtl_y1 = MtlArray{Float32}(edge.y1)

        Mtl_xx = MtlArray{Float32}(xx)
        Mtl_yy = MtlArray{Float32}(yy)

        n_e = nrow(edge)

        Mtl_vec0_x = MtlArray{Float32}(undef, n_x, n_y, n_e)
        Mtl_vec0_y = similar(Mtl_vec0_x)
        Mtl_vec1_x = similar(Mtl_vec0_x)
        Mtl_vec1_y = similar(Mtl_vec0_x)
        # Threadgroup size configuration
        threads_per_group = (10, 9, 9)  # = 810 < 832
        num_groups = (ceil(Int, n_x / threads_per_group[1]), 
                    ceil(Int, n_y / threads_per_group[2]), 
                    ceil(Int, n_e / threads_per_group[3]))

        # Kernel function execution
        @metal threads = threads_per_group groups = num_groups kernel_assign_1(Mtl_vec0_x, Mtl_vec0_y, 
                                                                                Mtl_vec1_x, Mtl_vec1_y, 
                                                                                Mtl_xx, Mtl_yy, 
                                                                                Mtl_x0, Mtl_y0, Mtl_x1, Mtl_y1)

        winding_angle = zeros(Float32, n_x, n_y)

        # ベクトルの外積
        Mtl_cross = Mtl_vec0_x .* Mtl_vec1_y .- Mtl_vec0_y .* Mtl_vec1_x

        # 2つのベクトルのなす角
        Mtl_dot = Mtl_vec0_x .* Mtl_vec1_x .+ Mtl_vec0_y .* Mtl_vec1_y
        Mtl_norm_0 = sqrt.(Mtl_vec0_x .* Mtl_vec0_x .+ Mtl_vec0_y .* Mtl_vec0_y)
        Mtl_norm_1 = sqrt.(Mtl_vec1_x .* Mtl_vec1_x .+ Mtl_vec1_y .* Mtl_vec1_y)
        Mtl_norm = Mtl_norm_0 .* Mtl_norm_1

        Mtl_div = MtlArray(min.(1.0f0, Mtl_dot ./ Mtl_norm))    # 要素の値が1以上の時は1に置換する
        Mtl_angle = acos.(Mtl_div)

    # https://www.omnicalculator.com/math/angle-between-two-vectors
        Mtl_theta = sign.(Mtl_cross) .* Mtl_angle
        # angle = πのときtheta = 0.0となるときがある 原因不明
        winding_angle .= @view sum(Array(Mtl_theta), dims = 3)[:, :, 1]

    # https://www.nttpc.co.jp/technology/number_algorithm.html
    # INSIDEの時，winding_angle >- 2π
    # OUTSIDEの時, winding_angle == 0
        if p == 1
            # INSIDEが1
            in_out[:, :, p_idx] .= (winding_angle .>=  1.99 * π)
            # in_out[:, :, p] = (winding_angle[:, :] .> 0.0)
        else
            # OUTSIDEが1
            # in_out[:, :, p] = (winding_angle[:, :] .< 0.1)
            in_out[:, :, p_idx] .= (winding_angle .< 1.1 * π)
            # in_out[:, :, p] = (winding_angle[:, :] .<= 0.0)
        end
    end

    subject_cal_arr = dropdims(prod(in_out, dims = 3), dims = 3)
    df_y = repeat(Vector{Int32}(1:n_y), inner = [n_x])
    df_x = repeat(Vector{Int32}(1:n_x), outer = [n_y])

    in_out_df =
    @chain DataFrame(subject_cal_arr, :auto) begin
        @pivot_longer(starts_with("x"), names_to = y, values_to = cal)
        # @pivot_longer(names_to = :y, values_to = :cal)
        insertcols!(:index_x => df_x, :index_y => df_y)
        @select(index_x, index_y, cal)
        # @select(:index_x, :index_y, :cal)
    end

    return in_out_df

end

# カーネル関数を使った3次元配列代入 #2_1
function kernel_assign_2_1(ab_x::MtlDeviceArray{Float32, 3}, ab_y::MtlDeviceArray{Float32, 3},
                            ll_cos_th::MtlDeviceArray{Float32, 1}, ll_sin_th::MtlDeviceArray{Float32, 1})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(ab_x, 1) || j > size(ab_x, 2) || k > size(ab_x, 3)
        return nothing
    end

    # ab
    # ab = bo - ao
    # bo = (ll * cos(θ) + xx, ll * sin(θ) + yy)
    # ao = (xx, yy)
    ab_x[i, j, k] = ll_cos_th[j]
    ab_y[i, j, k] = ll_sin_th[j]

    return nothing

end

# カーネル関数を使った3次元配列代入 #2_2
function kernel_assign_2_2(ac_x::MtlDeviceArray{Float32, 3}, ac_y::MtlDeviceArray{Float32, 3},
                            xx::MtlDeviceArray{Float32, 1}, yy::MtlDeviceArray{Float32, 1},
                            x0::MtlDeviceArray{Float32, 1}, y0::MtlDeviceArray{Float32, 1})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(ac_x, 1) || j > size(ac_x, 2) || k > size(ac_x, 3)

        return nothing

    end

    ac_x[i, j, k] = x0[k] - xx[i]
    ac_y[i, j, k] = y0[k] - yy[i]

    return nothing

end

# カーネル関数を使った3次元配列代入 #2_3
function kernel_assign_2_3(cd_x::MtlDeviceArray{Float32, 3}, cd_y::MtlDeviceArray{Float32, 3},
                            x0::MtlDeviceArray{Float32, 1}, y0::MtlDeviceArray{Float32, 1},
                            x1::MtlDeviceArray{Float32, 1}, y1::MtlDeviceArray{Float32, 1})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(cd_x, 1) || j > size(cd_x, 2) || k > size(cd_x, 3)

        return nothing

    end

    cd_x[i, j, k] = x1[k] - x0[k]
    cd_y[i, j, k] = y1[k] - y0[k]

    return nothing

end

# カーネル関数を使った交点を示すパラメータ(s, t)の計算
function kernel_s_t(s::MtlDeviceArray{Float32, 3}, t::MtlDeviceArray{Float32, 3},
                    ab_x::MtlDeviceArray{Float32, 3}, ab_y::MtlDeviceArray{Float32, 3},
                    ac_x::MtlDeviceArray{Float32, 3}, ac_y::MtlDeviceArray{Float32, 3},
                    ca_x::MtlDeviceArray{Float32, 3}, ca_y::MtlDeviceArray{Float32, 3},
                    cd_x::MtlDeviceArray{Float32, 3}, cd_y::MtlDeviceArray{Float32, 3})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(s, 1) || j > size(s, 2) || k > size(s, 3)
        return nothing
    end

    s[i, j, k] = (ac_x[i, j, k] * cd_y[i, j, k] - ac_y[i, j, k] * cd_x[i, j, k]) /
                (ab_x[i, j, k] * cd_y[i, j, k] - ab_y[i, j, k] * cd_x[i, j, k])
    t[i, j, k] = (ca_x[i, j, k] * ab_y[i, j, k] - ca_y[i, j, k] * ab_x[i, j, k]) /
                (cd_x[i, j, k] * ab_y[i, j, k] - cd_y[i, j, k] * ab_x[i, j, k])

    return nothing

end

# カーネル関数を使った有効な交点判定
function kernel_valid(s_valid::MtlDeviceArray{Bool, 3}, t_valid::MtlDeviceArray{Bool, 3},
                        s::MtlDeviceArray{Float32, 3}, t::MtlDeviceArray{Float32, 3})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(s_valid, 1) || j > size(s_valid, 2) || k > size(s_valid, 3)
        return nothing
    end

    s_valid[i, j, k] = (!isnan(s[i, j, k]) && s[i, j, k] >= 0 && s[i, j, k] <= 1)
    t_valid[i, j, k] = (t[i, j, k] >= 0 && t[i, j, k] <= 1)

    return nothing

end

# カーネル関数を使った有効な交点までの距離計算
function kernel_dist(dist::MtlDeviceArray{Float32, 3}, s::MtlDeviceArray{Float32, 3},
                    s_valid::MtlDeviceArray{Bool, 3}, t_valid::MtlDeviceArray{Bool, 3},
                    ll::Float32)::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(dist, 1) || j > size(dist, 2) || k > size(dist, 3)
        return nothing
    end

    dist[i, j, k] = (s[i, j, k] * s_valid[i, j, k] * t_valid[i, j, k]) * ll

    return nothing

end

# カーネル関数を使った2次元配列代入 #3
function kernel_assign_3(xx_arr::MtlDeviceArray{Float32, 2}, yy_arr::MtlDeviceArray{Float32, 2},
                            cos_arr::MtlDeviceArray{Float32, 2}, sin_arr::MtlDeviceArray{Float32, 2},
                            xx::MtlDeviceArray{Float32, 1}, yy::MtlDeviceArray{Float32, 1},
                            cos_th::MtlDeviceArray{Float32, 1}, sin_th::MtlDeviceArray{Float32, 1})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(xx_arr, 1) || j > size(xx_arr, 2)
        return nothing
    end

    xx_arr[i, j] = xx[i]
    yy_arr[i, j] = yy[i]
    cos_arr[i, j] = cos_th[j]
    sin_arr[i, j] = sin_th[j]

    return nothing

end

# カーネル関数を使った交点座標の計算
function kernel_crossing_xy(crossing_x::MtlDeviceArray{Float32, 2}, crossing_y::MtlDeviceArray{Float32, 2},
                            adj_x::MtlDeviceArray{Float32, 2}, adj_y::MtlDeviceArray{Float32, 2},
                            dist_crossing::MtlDeviceArray{Float32, 2}, adj_dist::MtlDeviceArray{Float32, 2},
                            cos_arr::MtlDeviceArray{Float32, 2}, sin_arr::MtlDeviceArray{Float32, 2},
                            xx_arr::MtlDeviceArray{Float32, 2}, yy_arr::MtlDeviceArray{Float32, 2})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(crossing_x, 1) || j > size(crossing_x, 2)
        return nothing
    end

    crossing_x[i, j] = dist_crossing[i, j] * cos_arr[i, j] + xx_arr[i, j]
    crossing_y[i, j] = dist_crossing[i, ] * sin_arr[i, j] + yy_arr[i, j]
    adj_x[i, j] = adj_dist[i, j] * cos_arr[i, j] + xx_arr[i, j]
    adj_y[i, j] = adj_dist[i, j] * sin_arr[i, j] + yy_arr[i, j]

    return nothing
    
end

# カーネル関数を使った多角形の面積計算（座標法）
function kernel_shoelace(isovist_partial::MtlDeviceArray{Float32, 2}, A_Index_partial::MtlDeviceArray{Float32, 2},
                            crossing_x::MtlDeviceArray{Float32, 2}, crossing_y::MtlDeviceArray{Float32, 2},
                            adj_x::MtlDeviceArray{Float32, 2}, adj_y::MtlDeviceArray{Float32, 2})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(isovist_partial, 1) || j > size(isovist_partial, 2)
        return nothing
    end

    isovist_partial[i, j] = (crossing_x[i, j] * crossing_y[i, mod1(j + 1, size(crossing_y, 2))] -
                            crossing_x[i, mod1(j + 1, size(crossing_x, 2))] * crossing_y[i, j]) / 2.0f0

    A_Index_partial[i, j] = (adj_x[i, j] * adj_y[i, mod1(j + 1, size(adj_y, 2))] -
                            adj_x[i, mod1(j + 1, size(adj_x, 2))] * adj_y[i, j]) / 2.0f0

    return nothing
    
end

# 指標値計算
indeces = function (all_points_df, edge_df)

    # 外周および遮蔽物を構成する辺の数
    num_edges = nrow(edge_df)
    Mtl_x0 = MtlArray{Float32}(edge_df.x0) # OK
    Mtl_y0 = MtlArray{Float32}(edge_df.y0) # OK
    Mtl_x1 = MtlArray{Float32}(edge_df.x1) # OK
    Mtl_y1 = MtlArray{Float32}(edge_df.y1) # OK

    # 計算実施点を取り出す
    not_subject_df = @chain all_points_df begin
        @filter(cal != 1)
    end

    subject_df = @chain all_points_df begin
        @filter(cal == 1)
    end

    Mtl_xx = MtlArray{Float32}(subject_df.xx) # OK
    Mtl_yy = MtlArray{Float32}(subject_df.yy) # OK
    num_points = size(Mtl_xx, 1)

    # 半直線方向ベクトル用定数
    ll = Float32(9999)

    # スキャン角分割数
    # 余裕率を考慮して，分割数を決定
    # α = Float32(0.5) # OK
    α = Float32(0.5)
    div_n = @chain Metal.current_device() begin
        _.maxBufferLength
        floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
        convert(Int, _)
    end

    # [点, 角度, 辺]
    Mtl_ab_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    Mtl_ab_y = similar(Mtl_ab_x)

    cos_th = @chain collect(0:div_n - 1) begin
        _ .* (2 * π / div_n)
        cos.(_)
        convert.(Float32, _)
    end
    Mtl_cos_th = MtlArray{Float32}(cos_th)
    Mtl_ll_cos_th = ll .* Mtl_cos_th

    sin_th = @chain collect(0:div_n - 1) begin
        _ .* (2 * π / div_n)
        sin.(_)
        convert.(Float32, _)
    end
    Mtl_sin_th = MtlArray{Float32}(sin_th)
    Mtl_ll_sin_th = ll .* Mtl_sin_th

    # Threadgroup size configuration
    # threads_per_group = (10, 10, 10)  # MacBook Pro M4Max OK, Mac Studio NG
    threads_per_group = (8, 8, 4)  #  <= 512
    num_groups = (ceil(Int, num_points / threads_per_group[1]), 
                ceil(Int, div_n / threads_per_group[2]), 
                ceil(Int, num_edges / threads_per_group[3]))
    # Kernel function execution
    @metal threads = threads_per_group groups = num_groups kernel_assign_2_1(Mtl_ab_x, Mtl_ab_y, 
                                                                            Mtl_ll_cos_th, Mtl_ll_sin_th)

    # [点, 角度, 辺]
    Mtl_ac_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    Mtl_ac_y = similar(Mtl_ac_x)
    Mtl_cd_x = similar(Mtl_ac_x)
    Mtl_cd_y = similar(Mtl_ac_x)
    Mtl_ca_x = similar(Mtl_ac_x)
    Mtl_ca_y = similar(Mtl_ac_x)

    # Kernel function execution
    @metal threads = threads_per_group groups = num_groups kernel_assign_2_2(Mtl_ac_x, Mtl_ac_y, 
                                                                            Mtl_xx, Mtl_yy,
                                                                            Mtl_x0, Mtl_y0)
ac_y = Array(Mtl_ac_y)
@show ac_y
    Mtl_ca_x .= -Mtl_ac_x
    Mtl_ca_y .= -Mtl_ac_y

    @metal threads = threads_per_group groups = num_groups kernel_assign_2_3(Mtl_cd_x, Mtl_cd_y, 
                                                                            Mtl_x0, Mtl_y0,
                                                                            Mtl_x1, Mtl_y1)

    # 交点を示すパラメータ(s, t)
    # 視線ベクトルと辺が平行な時，外積は0になる
    Mtl_s = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    Mtl_t = similar(Mtl_s)
    @metal threads = threads_per_group groups = num_groups kernel_s_t(Mtl_s, Mtl_t,
                                                                    Mtl_ab_x, Mtl_ab_y,
                                                                    Mtl_ac_x, Mtl_ac_y,
                                                                    Mtl_ca_x, Mtl_ca_y,
                                                                    Mtl_cd_x, Mtl_cd_y)

s = Array(Mtl_s) 
@show Mtl_s

    # # 有効な交点との距離
    # Mtl_s_valid = similar(Mtl_s, Bool)
    # Mtl_t_valid = similar(Mtl_s, Bool)
    # @metal threads = threads_per_group groups = num_groups kernel_valid(Mtl_s_valid, Mtl_t_valid, Mtl_s, Mtl_t)

    # Mtl_dist_crossing = MtlArray{Float32}(undef, num_points, div_n, num_edges)  # similar(Mtl_ab_x)
    # @metal threads = threads_per_group groups = num_groups kernel_dist(Mtl_dist_crossing, Mtl_s,
    #                                                                     Mtl_s_valid, Mtl_t_valid, ll)

    # # 計算実施点から遮蔽物（外周）までの距離
    # # 計算実施点は必ずll未満の視線距離を有するので
    # # 視線と考慮対象エッジが平行な場合(NaN)および交点を持たない場合(0 or -0)は
    # # llとする ← 最小距離にはならないはず
    # # 平行 -> NaN 平行ではなく交点を持たない -> 0
    # # Mtl_dist_crossing: 考慮点番号, 角度，辺番号
    # # Mtl_min_dist_crossing: 考慮点番号, 角度
    # min_dist_crossing =
    # @chain Array(Mtl_dist_crossing) begin
    #     # @aside @show _[1, 4, :]
    #     replace(0.0 => ll)
    #     replace(-0.0 => ll)
    #     replace(NaN => ll)
    #     minimum(dims = 3)
    #     reshape(num_points, div_n)
    # end

    # # A-Index用に低減関数で調整した距離を求める
    # ν = 1.1915889
    # d = TDist(ν)
    # adj_dist = similar(min_dist_crossing)
    # @. adj_dist = cdf(d, min_dist_crossing) - 0.5f0 # cdf(d, 0) = 0.5

    # # Δ = Float32(0.5f0)
    # # # Δ = 0.001f0
    # # Mtl_steps =MtlArray{Int32}(undef, num_points, div_n)
    # # @. Mtl_steps = unsafe_trunc(Int32, Mtl_min_dist_crossing / Δ)
    # # Mtl_start_p = Metal.zeros(Float32, num_points, div_n)

    # # Mtl_adj_dist = similar(Mtl_min_dist_crossing)
    # # threads_per_group = (32, 32)
    # # num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    # # ## カーネル関数（数値積分）を実行
    # # @metal threads = threads_per_group groups = num_groups kernel_integral(Mtl_start_p, Mtl_adj_dist, Mtl_steps, Δ)

    # Mtl_xx_arr = MtlArray{Float32}(undef, num_points, div_n)
    # Mtl_yy_arr = similar(Mtl_xx_arr)
    # Mtl_cos_arr = similar(Mtl_xx_arr)
    # Mtl_sin_arr = similar(Mtl_xx_arr)

    # threads_per_group = (32, 32)
    # num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))
    # @metal threads = threads_per_group groups = num_groups kernel_assign_3(Mtl_xx_arr, Mtl_yy_arr, 
    #                                                                     Mtl_cos_arr, Mtl_sin_arr,
    #                                                                     Mtl_xx, Mtl_yy,
    #                                                                     Mtl_cos_th, Mtl_sin_th)

    # # カーネル関数を使った交点座標の計算
    # # 座標番号, 角度
    # Mtl_min_dist_crossing = MtlArray(min_dist_crossing)
    # Mtl_adj_dist = MtlArray(adj_dist)
    # Mtl_crossing_x = similar(Mtl_min_dist_crossing)
    # Mtl_crossing_y = similar(Mtl_min_dist_crossing)
    # Mtl_adj_x = similar(Mtl_adj_dist)
    # Mtl_adj_y = similar(Mtl_adj_dist)

    # threads_per_group = (16, 16)    # <= 832
    # num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))
    # @metal threads = threads_per_group groups = num_groups kernel_crossing_xy(Mtl_crossing_x, Mtl_crossing_y,
    #                                                                     Mtl_adj_x, Mtl_adj_y,
    #                                                                     Mtl_min_dist_crossing, Mtl_adj_dist,
    #                                                                     Mtl_cos_arr, Mtl_sin_arr,
    #                                                                     Mtl_xx_arr, Mtl_yy_arr)

    # # 座標法による多角形面積計算
    # Mtl_isovist_partial = similar(Mtl_crossing_x)
    # Mtl_A_Index_partial = similar(Mtl_crossing_x)

    # ## カーネル関数（座標法）を実行
    # # threads_per_group = (28, 27)    # <= 768
    # # num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))
    # @metal threads = threads_per_group groups = num_groups kernel_shoelace(Mtl_isovist_partial, Mtl_A_Index_partial,
    #                                                                     Mtl_crossing_x, Mtl_crossing_y,
    #                                                                     Mtl_adj_x, Mtl_adj_y)

    # isovist_vec =
    # @chain Mtl_isovist_partial begin
    #     sum(dims = 2)
    #     reshape(num_points, 1)
    #     vec(Array(_))
    # end

    # A_Index_vec =
    # @chain Mtl_A_Index_partial begin
    #     sum(dims = 2)
    #     reshape(num_points, 1)
    #     vec(Array(_))
    # end

    # index_df = DataFrame(
    #     x = subject_df.xx,
    #     y = subject_df.yy,
    #     isovist = isovist_vec,
    #     a_index = A_Index_vec
    # )

    # return (index_df)
    
    # @show (min_dist_crossing[1, 5], size(min_dist_crossing))
    # 多角形の面積の計算
    # PolygonOps.area
    # https://docs.juliahub.com/PolygonOps/TUelr/0.1.2/
end

# 座標データファイル名指定
f_name = "25F_ALL"
f_path = string("../Plan/", f_name, ".csv")

# 外周および遮蔽物の生成
outer, obs = reading_coordinate(f_path)
# outer = generate_outer()
# outer_id_max = maximum(outer.id)
xmin_outer = minimum(outer.x0)
xmax_outer = maximum(outer.x0)
ymin_outer = minimum(outer.y0)
ymax_outer = maximum(outer.y0)

# 遮蔽物生成
# obs = generate_obs(outer_id_max)

# 外周及び遮蔽物を構成する
# 辺を取得
edge_outer = polygon_to_edges(outer)
edge_obs = polygon_to_edges(obs)
edge_df = vcat(edge_outer, edge_obs)

# 計算座標間隔
# 空間分割数
const s_div = 200
dd_x = (xmax_outer - xmin_outer) / s_div
dd_y = (ymax_outer - ymin_outer) / s_div

xx = Vector{Float32}(undef, s_div + 1)
yy = Vector{Float32}(undef, s_div + 1)
xx[:] = [xmin_outer + (i - 1) * dd_x for i in 1:length(xx)]
yy[:] = [ymin_outer + (i - 1) * dd_y for i in 1:length(yy)]

# 内外判定
# PolygonOps,inpolygon
# https://docs.juliahub.com/PolygonOps/TUelr/0.1.2/
subject_cal_df = 
@chain in_out_judge(xx, yy, edge_df) begin
    @mutate(
        xx = (!!xmin_outer + (index_x - 1) * !!dd_x),
        yy = (!!ymin_outer + (index_y - 1) * !!dd_y)
    )
end

# 指標格納データフレーム
# index_df = DataFrame(
#     x = Float32[], 
#     y = Float32[], 
#     isovist = Float32[], 
#     a_index = Float32[]
# )
index_df = indeces(subject_cal_df, edge_df)
# CSV.write(string("JL_", f_name, "_index.csv"), index_df)

#= 
@rput subject_cal_df
@rput xmin_outer
@rput ymin_outer
@rput xmax_outer
@rput ymax_outer
R"
library(tidyverse)

subject_cal_df =
    subject_cal_df %>%
    mutate(
        cal = case_when(
            cal == \"FALSE\" ~ 0,
            cal == \"TRUE\" ~ 1
        )
    ) %>%
    write_csv(\"subject_cal_df.csv\")

# # p <- 
# #     subject_cal_df %>%
# #     ggplot(aes(x = xx, y = yy, z = cal)) +
# #     stat_contour_filled(bins = 15) +
# #     theme(aspect.ratio = (ymax_outer - ymin_outer) / (xmax_outer - xmin_outer))
# # plot(p)
" =#

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


ac_y = Float32[19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.393 19.39

Excessive output truncated after 524504 bytes.

ErrorException: Scalar indexing is disallowed.
Invocation of getindex resulted in scalar indexing of a GPU array.
This is typically caused by calling an iterating implementation of a method.
Such implementations *do not* execute on the GPU, but very slowly on the CPU,
and therefore should be avoided.

If you want to allow scalar iteration, use `allowscalar` or `@allowscalar`
to enable scalar iteration globally or for the operations in question.